<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/xgboost_fullstack_trading_pipeline_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get remove --purge -y cuda* libcuda* nvidia* || echo "No conflicting CUDA packages"
!apt-get autoremove -y
!apt-get clean

In [2]:
#Protocol Buffer Fix (for TensorFlow)
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

In [3]:
#Update Colab Environment and System Libraries
!apt-get update -y && apt-get upgrade -y


In [4]:
#Install Correct Version of CUDA for Colab GPU
!apt-get update -qq && apt-get install -y \
    libcusolver11 libcusparse11 libcurand10 libcufft10 libnppig10 libnppc10 libnppial10 \
    cuda-toolkit-12-4

In [5]:
#Set Correct CUDA Paths
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.4'
os.environ['PATH'] += ':/usr/local/cuda-12.4/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-12.4/lib64'


In [6]:
#Install RAPIDS and NVIDIA Dependencies
!pip install --extra-index-url=https://pypi.nvidia.com \
    cuml-cu12==25.2.0 cudf-cu12==25.2.0 cupy-cuda12x dask-cuda==25.2.0 dask-cudf-cu12==25.2.0


In [7]:
#Install TensorFlow (latest GPU-compatible version)
!pip install tensorflow==2.18.0

#Install Stable Baselines3 and Trading Libraries
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib

#Install Miscellaneous Libraries
!pip install matplotlib scikit-learn pandas numba==0.61.0

#Install PyTorch with GPU Support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [8]:
#Install TensorFlow (latest GPU-compatible version)
!pip install tensorflow==2.18.0


import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU memory growth enabled")
    except RuntimeError as e:
        print(f"TensorFlow GPU memory config failed: {e}")


In [9]:
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance --quiet
!pip install stable-baselines3[extra] --quiet


In [10]:
#Import Required Libraries
import gc
import json
import os
import random
import time
from collections import deque
from datetime import datetime

import cupy as cp
import cudf
import cuml
import dask
import gymnasium as gym
import gym_anytrading
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
import yfinance as yf
from cuml.ensemble import RandomForestClassifier
from gym_anytrading.envs import StocksEnv
from gymnasium.spaces import Box
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

#Ticker List and CONFIG
ticker_list = [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL'
]

strategy_name = "sac_ppo_td3_multi_stock_v1"

CONFIG = {
    'symbols': [],
    'period': '720d',
    'interval': '1h',
    'target': 'Target',
    'sharpe_threshold': 1.5,
    'return_threshold': 1.25,
    'strategy_name': strategy_name
}
#Import Required Libraries
import gc
import json
import os
import random
import time
from collections import deque
from datetime import datetime

import cupy as cp
import cudf
import cuml
import dask
import gymnasium as gym
import gym_anytrading
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
import yfinance as yf
from cuml.ensemble import RandomForestClassifier
from gym_anytrading.envs import StocksEnv
from gymnasium.spaces import Box
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

#Ticker List and CONFIG
ticker_list = [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL'
]


def download_stock_data(ticker, period="720d", interval="1h", max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading {ticker}...")
            df = yf.download(ticker, period=period, interval=interval)
            if not df.empty:
                df.reset_index(inplace=True)
                df['Symbol'] = ticker
                return df
            raise ValueError("Empty data")
        except Exception as e:
            print(f"Error: {e}. Retrying in {attempt * 5} sec...")
            time.sleep(attempt * 5)
    print(f"Failed to download {ticker}")
    return None

#Feature Engineering Function
def compute_enhanced_features(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.loc[:, ~df.columns.duplicated()]

    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['STD_20'] = df['Close'].rolling(20).std()
    df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
    df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']
    df['Lowest_Low'] = df['Low'].rolling(14).min()
    df['Highest_High'] = df['High'].rolling(14).max()
    denom = (df['Highest_High'] - df['Lowest_Low']).replace(0, np.nan)
    df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / denom) * 100
    df['ROC'] = df['Close'].pct_change(10)
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).cumsum()
    typical_price = (df['High'] + df['Low'] + df['Close']) / 3
    df['CCI'] = (typical_price - typical_price.rolling(20).mean()) / (0.015 * typical_price.rolling(20).std())
    df['PROC'] = ((df['Close'] - df['Close'].shift(12)) / df['Close'].shift(12)) * 100
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['Expanding_Mean'] = df['Close'].expanding().mean()
    df['EMA_10'] = df['Close'].ewm(span=10).mean()
    df['EMA_50'] = df['Close'].ewm(span=50).mean()
    df['MACD_Line'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['MACD_Signal'] = df['MACD_Line'].ewm(span=9).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['MACD_Signal']
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    df['True_Range'] = df[['High', 'Low', 'Close']].apply(
        lambda x: max(x.iloc[0] - x.iloc[1], abs(x.iloc[0] - x.iloc[2]), abs(x.iloc[1] - x.iloc[2])), axis=1)
    df['ATR'] = df['True_Range'].rolling(14).mean()
    df['+DM'] = np.where((df['High'].diff() > df['Low'].diff()) & (df['High'].diff() > 0), df['High'].diff(), 0)
    df['-DM'] = np.where((df['Low'].diff() > df['High'].diff()) & (df['Low'].diff() > 0), df['Low'].diff(), 0)
    df['+DI'] = 100 * df['+DM'].rolling(14).mean() / df['ATR']
    df['-DI'] = 100 * df['-DM'].rolling(14).mean() / df['ATR']
    df['ADX'] = abs(df['+DI'] - df['-DI']).rolling(14).mean()
    df['Volume_Avg'] = df['Volume'].rolling(20).mean()
    df['Volume_Change'] = df['Volume'].pct_change()
    df['Volume_Change_MA'] = df['Volume_Change'].rolling(10).mean()
    df['Volume_Change_Ratio'] = df['Volume_Change'] / df['Volume_Change'].shift(1)
    df['Relative_Volume'] = df['Volume'] / df['Volume_Avg']
    df['Trailing_Stop'] = np.minimum(df['Close'] * 0.985, df['Close'] - (df['ATR'] * 0.3))
    df['Buy_Signal'] = np.where((df['RSI'] < 60) & (df['EMA_10'] > df['EMA_50']) &
                                ((df['MACD_Line'] > df['MACD_Signal']) | (df['MACD_Line'].diff() > 0)) &
                                (df['Volume'] > (0.4 * df['Volume_Avg'])) & (df['ADX'] > 18), 1, 0)
    df['Sell_Signal'] = np.where(((df['EMA_10'] < df['EMA_50']) & (df['RSI'] > 60)) |
                                 ((df['MACD_Line'] < df['MACD_Signal']) & (df['RSI'] > 65)) |
                                 (df['Close'] < df['Trailing_Stop']) |
                                 ((df['Volume'] > 0.5 * df['Volume_Avg']) & (df['ADX'] > 20)), 1, 0)
    df['Sell_Signal_Debug'] = np.where(((df['MACD_Hist'] < 0.5) | (df['MACD_Line'] < df['MACD_Signal'])) &
                                       (df['RSI'] < 55) & (df['ADX'] > 15) &
                                       ((df['Close'] < df['Trailing_Stop']) | (df['EMA_10'] < df['EMA_50'])) &
                                       (df['Volume'] > 0.5 * df['Volume_Avg']), 1, 0)
    df['Future_Close'] = df['Close'].shift(-10)
    df['Volatility'] = df['Close'].pct_change().rolling(window=20).std()
    df['Return'] = (df['Future_Close'] - df['Close']) / df['Close']
    df['Target'] = np.select([df['Return'] > 0.02, df['Return'] < -0.02], [1, -1], default=0)
    df['Multi_Class_Target'] = df['Target']
    df['Hour'] = pd.to_datetime(df['Datetime']).dt.hour
    df['DayOfWeek'] = pd.to_datetime(df['Datetime']).dt.dayofweek
    df['Session'] = np.where((df['Hour'] >= 9) & (df['Hour'] <= 16), 'Regular',
                             np.where((df['Hour'] < 9), 'Pre-market', 'After-hours'))
    df['MACD_Crossover'] = np.where(df['MACD_Line'] > df['MACD_Signal'], 1, 0)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

all_dfs = []

for ticker in ticker_list:
    df_single = download_stock_data(ticker, period=CONFIG['period'], interval=CONFIG['interval'])
    if df_single is not None:
        try:
            df_features = compute_enhanced_features(df_single)
            all_dfs.append(df_features)
        except Exception as e:
            print(f"Feature engineering failed for {ticker}: {e}")
    else:
        print(f"Failed to download {ticker}")

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    print(f"Combined dataset created with shape: {df.shape}")
else:
    df = pd.DataFrame()
    print("No data available.")

if not df.empty:
    df.to_csv("multi_stock_feature_engineered_dataset.csv", index=False)
    print("Saved locally to multi_stock_feature_engineered_dataset.csv")

    drive_path = "/content/drive/MyDrive/trading_data/"
    os.makedirs(drive_path, exist_ok=True)
    df.to_csv(os.path.join(drive_path, "multi_stock_feature_engineered_dataset.csv"), index=False)
    print(f"Also saved to Google Drive at {drive_path}multi_stock_feature_engineered_dataset.csv")


#Download Function
def download_stock_data(ticker, period="720d", interval="1h", max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading {ticker}...")
            df = yf.download(ticker, period=period, interval=interval)
            if not df.empty:
                df.reset_index(inplace=True)
                df['Symbol'] = ticker
                return df
            raise ValueError("Empty data")
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            time.sleep(attempt * 5)
    print(f"Failed to download {ticker}")
    return None

#Feature Engineering
def compute_enhanced_features(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.loc[:, ~df.columns.duplicated()]

    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['STD_20'] = df['Close'].rolling(20).std()
    df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
    df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']
    df['Lowest_Low'] = df['Low'].rolling(14).min()
    df['Highest_High'] = df['High'].rolling(14).max()
    denom = (df['Highest_High'] - df['Lowest_Low']).replace(0, np.nan)
    df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / denom) * 100
    df['ROC'] = df['Close'].pct_change(10)
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).cumsum()
    typical_price = (df['High'] + df['Low'] + df['Close']) / 3
    df['CCI'] = (typical_price - typical_price.rolling(20).mean()) / (0.015 * typical_price.rolling(20).std())
    df['PROC'] = ((df['Close'] - df['Close'].shift(12)) / df['Close'].shift(12)) * 100
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['Expanding_Mean'] = df['Close'].expanding().mean()
    df['EMA_10'] = df['Close'].ewm(span=10).mean()
    df['EMA_50'] = df['Close'].ewm(span=50).mean()
    df['MACD_Line'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['MACD_Signal'] = df['MACD_Line'].ewm(span=9).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['MACD_Signal']
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    df['True_Range'] = df[['High', 'Low', 'Close']].apply(
        lambda x: max(x.iloc[0] - x.iloc[1], abs(x.iloc[0] - x.iloc[2]), abs(x.iloc[1] - x.iloc[2])), axis=1)
    df['ATR'] = df['True_Range'].rolling(14).mean()
    df['+DM'] = np.where((df['High'].diff() > df['Low'].diff()) & (df['High'].diff() > 0), df['High'].diff(), 0)
    df['-DM'] = np.where((df['Low'].diff() > df['High'].diff()) & (df['Low'].diff() > 0), df['Low'].diff(), 0)
    df['+DI'] = 100 * df['+DM'].rolling(14).mean() / df['ATR']
    df['-DI'] = 100 * df['-DM'].rolling(14).mean() / df['ATR']
    df['ADX'] = abs(df['+DI'] - df['-DI']).rolling(14).mean()
    df['Volume_Avg'] = df['Volume'].rolling(20).mean()
    df['Volume_Change'] = df['Volume'].pct_change()
    df['Volume_Change_MA'] = df['Volume_Change'].rolling(10).mean()
    df['Volume_Change_Ratio'] = df['Volume_Change'] / df['Volume_Change'].shift(1)
    df['Relative_Volume'] = df['Volume'] / df['Volume_Avg']
    df['Trailing_Stop'] = np.minimum(df['Close'] * 0.985, df['Close'] - (df['ATR'] * 0.3))
    df['Buy_Signal'] = np.where((df['RSI'] < 60) & (df['EMA_10'] > df['EMA_50']) &
                                ((df['MACD_Line'] > df['MACD_Signal']) | (df['MACD_Line'].diff() > 0)) &
                                (df['Volume'] > (0.4 * df['Volume_Avg'])) & (df['ADX'] > 18), 1, 0)
    df['Sell_Signal'] = np.where(((df['EMA_10'] < df['EMA_50']) & (df['RSI'] > 60)) |
                                 ((df['MACD_Line'] < df['MACD_Signal']) & (df['RSI'] > 65)) |
                                 (df['Close'] < df['Trailing_Stop']) |
                                 ((df['Volume'] > 0.5 * df['Volume_Avg']) & (df['ADX'] > 20)), 1, 0)
    df['Sell_Signal_Debug'] = np.where(((df['MACD_Hist'] < 0.5) | (df['MACD_Line'] < df['MACD_Signal'])) &
                                       (df['RSI'] < 55) & (df['ADX'] > 15) &
                                       ((df['Close'] < df['Trailing_Stop']) | (df['EMA_10'] < df['EMA_50'])) &
                                       (df['Volume'] > 0.5 * df['Volume_Avg']), 1, 0)
    df['Future_Close'] = df['Close'].shift(-10)
    df['Volatility'] = df['Close'].pct_change().rolling(window=20).std()
    df['Return'] = (df['Future_Close'] - df['Close']) / df['Close']
    df['Target'] = np.select([df['Return'] > 0.02, df['Return'] < -0.02], [1, -1], default=0)
    df['Multi_Class_Target'] = df['Target']
    df['Hour'] = pd.to_datetime(df['Datetime']).dt.hour
    df['DayOfWeek'] = pd.to_datetime(df['Datetime']).dt.dayofweek
    df['Session'] = np.where((df['Hour'] >= 9) & (df['Hour'] <= 16), 'Regular',
                             np.where((df['Hour'] < 9), 'Pre-market', 'After-hours'))
    df['MACD_Crossover'] = np.where(df['MACD_Line'] > df['MACD_Signal'], 1, 0)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df


Combined dataset created with shape: (262116, 51)
Saved locally to multi_stock_feature_engineered_dataset.csv
Also saved to Google Drive at /content/drive/MyDrive/trading_data/multi_stock_feature_engineered_dataset.csv


In [11]:
!rm -rf /content/drive

In [15]:
# === Imports ===
import os
import gc
import json
import joblib
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from google.colab import drive
import pywt

# === Mount Google Drive ===
drive.mount("/content/drive", force_remount=True)

# === Configurations ===
SAVE_DIR = "/content/drive/MyDrive/QuantConnect/results_lightgbm/xgb_walkforward_results"
RESULTS_DIR = "/content/drive/MyDrive/QuantConnect/results_xgb_walkforward_results"
FINAL_MODEL_DIR = f"{RESULTS_DIR}/models"
os.makedirs(f"{RESULTS_DIR}/plots", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/data", exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# === Load Data ===
df = pd.read_csv("multi_stock_feature_engineered_dataset.csv")
df['Datetime'] = pd.to_datetime(df['Datetime']).dt.tz_localize(None)

# === Denoising ===
def denoise_signal(signal):
    coeffs = pywt.wavedec(signal, 'db4', level=2)
    coeffs[1:] = [np.zeros_like(c) for c in coeffs[1:]]
    return pywt.waverec(coeffs, 'db4')[:len(signal)]

df['Close'] = denoise_signal(df['Close'].values)

# === Add Market Regime ===
df['Volatility'] = df['Close'].pct_change().rolling(20).std()
df['Regime'] = np.where(df['Volatility'] < 0.01, 'Bull',
                        np.where(df['Volatility'] > 0.03, 'Bear', 'Sideways'))
df = pd.concat([df, pd.get_dummies(df['Regime'], prefix="Regime")], axis=1)

# === Feature Columns ===
features = ['SMA_20', 'STD_20', 'Upper_Band', 'Lower_Band', 'Stoch',
            'Regime_Bull', 'Regime_Bear', 'Regime_Sideways']
target = "Target"
label_map = {0: 0, 1: 1, -1: 2}  # Ensure mapping if needed

results = []

# === Ticker List ===
test_mode = True
simulate_latency = True
TICKERS = ['AAPL'] if test_mode else [...]

# === Rolling Window Generator ===
def generate_date_windows(start_date, end_date, train_days=365, test_days=60, step_days=60):
    windows = []
    current = pd.to_datetime(start_date)
    while current + timedelta(days=train_days + test_days) <= pd.to_datetime(end_date):
        train_start = current
        train_end = train_start + timedelta(days=train_days)
        test_start = train_end
        test_end = test_start + timedelta(days=test_days)
        windows.append((train_start, train_end, test_start, test_end))
        current += timedelta(days=step_days)
    return windows

# === Walkforward XGB ===
def walkforward_xgb(df_ticker, ticker):
    df_ticker = df_ticker.dropna(subset=features + [target]).copy()
    df_ticker['Target_Mapped'] = df_ticker[target].map(label_map)
    windows = generate_date_windows("2020-01-01", "2024-01-01")

    last_model = None
    last_valid = False

    for (train_start, train_end, test_start, test_end) in windows:
        print(f"\nTraining: {train_start.date()} to {train_end.date()}, Testing: {test_start.date()} to {test_end.date()}")

        train_df = df_ticker[(df_ticker['Datetime'] >= train_start) & (df_ticker['Datetime'] < train_end)]
        test_df = df_ticker[(df_ticker['Datetime'] >= test_start) & (df_ticker['Datetime'] < test_end)]

        if len(train_df) < 200 or len(test_df) < 50:
            print("Skipping due to insufficient data")
            continue

        print(train_df['Target_Mapped'].value_counts())

        X_train, y_train = train_df[features], train_df['Target_Mapped']
        X_test, y_test = test_df[features], test_df['Target_Mapped']

        model = XGBClassifier(n_estimators=100, learning_rate=0.05, tree_method='hist', random_state=42)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)
        preds = np.argmax(proba, axis=1)
        conf_scores = proba.max(axis=1)

        signal = preds - 1
        confidence_threshold = 0.6
        signal[(conf_scores < confidence_threshold)] = 0
        test_df = test_df.copy()
        test_df['Signal'] = signal

        print("Signal distribution:", test_df['Signal'].value_counts().to_dict())
        print("Confidence: min = %.3f  max = %.3f  mean = %.3f" % (conf_scores.min(), conf_scores.max(), conf_scores.mean()))
        print("Missed signals due to low confidence:", sum(conf_scores < confidence_threshold))

        test_df['Rolling_Mean_50'] = test_df['Close'].rolling(50).mean()

        capital = 100000
        shares = 0
        portfolio = []
        slippage_pct = 0.001
        last_trade_idx = -10

        for i, row in test_df.iterrows():
            price = row['Close']
            atr = row.get('ATR', 2.0)
            signal = row['Signal']

            if simulate_latency:
                time.sleep(random.uniform(0.1, 0.4))

            if signal == 1 and (i - last_trade_idx > 5) and capital > price and price > row['Rolling_Mean_50']:
                risk_per_trade = 0.02 * capital
                stop_price = price - atr
                risk_per_share = max(price - stop_price, 1e-6)
                qty = min((risk_per_trade // risk_per_share), (capital * 0.1) // price)
                buy_price = price * (1 + slippage_pct)

                if qty > 0:
                    capital -= qty * buy_price
                    shares += qty
                    last_trade_idx = i
                    print(f"[Broker] BUY {qty:.1f} {ticker} @ {buy_price:.2f}")

            elif signal == -1 and shares > 0 and (i - last_trade_idx > 5):
                sell_price = price * (1 - slippage_pct)
                capital += shares * sell_price
                print(f"[Broker] SELL {shares:.1f} {ticker} @ {sell_price:.2f}")
                shares = 0
                last_trade_idx = i

            portfolio.append(capital + shares * price)

        if not portfolio or np.std(portfolio) == 0:
            print("No trades or zero variance portfolio. Attempting fallback...")
            if last_valid:
                proba = last_model.predict_proba(X_test)
                preds = np.argmax(proba, axis=1)
                conf_scores = proba.max(axis=1)
                signal = preds - 1
                signal[(conf_scores < confidence_threshold)] = 0
                test_df['Signal'] = signal
                print("Reused last model. New signals:", test_df['Signal'].value_counts().to_dict())
                continue
            else:
                print(" No fallback model available.")
                continue

        final_value = portfolio[-1]
        return_pct = (final_value - 100000) / 100000 * 100
        returns = pd.Series(portfolio).pct_change().fillna(0)
        sharpe = (returns.mean() / (returns.std() + 1e-6)) * np.sqrt(252)
        drawdown = ((pd.Series(portfolio).cummax() - pd.Series(portfolio)) / pd.Series(portfolio).cummax()).max() * 100

        results.append({
            "Ticker": ticker,
            "Train Period": f"{train_start.date()} to {train_end.date()}",
            "Test Period": f"{test_start.date()} to {test_end.date()}",
            "Model": "XGBoost",
            "Accuracy": round(accuracy_score(y_test, preds), 4),
            "Sharpe": round(sharpe, 3),
            "Drawdown": round(drawdown, 2),
            "Return": round(return_pct, 2),
            "Final_Portfolio": round(final_value, 2)
        })

        filename_prefix = f"{ticker}_{train_start.date()}_{test_start.date()}"
        joblib.dump(model, os.path.join(FINAL_MODEL_DIR, f"xgb_{filename_prefix}.pkl"))
        with open(os.path.join(FINAL_MODEL_DIR, f"xgb_{filename_prefix}_features.json"), "w") as f:
            json.dump(features, f)

        test_df.to_csv(os.path.join(RESULTS_DIR, "data", f"{ticker}_{filename_prefix}_result.csv"), index=False)

        plt.figure(figsize=(12, 6))
        plt.plot(test_df['Close'].values, label='Close Price')
        plt.plot(test_df['Signal'].cumsum(), label='Cumulative Signal')
        plt.title(f"{ticker} - XGB Filtered Strategy Signals")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, "plots", f"{ticker}_{filename_prefix}_xgb_signals.png"))
        plt.close()

        last_model = model
        last_valid = True

# === Run All Tickers ===
for ticker in TICKERS:
    print(f"\n Processing {ticker}")
    df_ticker = df[df['Symbol'] == ticker].copy()
    if len(df_ticker) < 1000:
        print(f" Skipping {ticker}, not enough data.")
        continue
    walkforward_xgb(df_ticker, ticker)
    gc.collect()

# === Save Summary ===
summary_df = pd.DataFrame(results)
summary_df.to_csv(os.path.join(RESULTS_DIR, "xgb_walkforward_metrics.csv"), index=False)

summary_df['score'] = (
    summary_df['Sharpe'] * 0.4 +
    summary_df['Return'] * 0.3 +
    summary_df['Final_Portfolio'] * 0.3
)
summary_df.to_csv(os.path.join(RESULTS_DIR, "xgb_model_selector_metrics.csv"), index=False)

best_models = summary_df.sort_values(['Ticker', 'score'], ascending=[True, False])\
                        .groupby('Ticker').first().reset_index()
best_models.to_excel(os.path.join(RESULTS_DIR, "xgb_best_models_by_score.xlsx"), index=False)



 Processing AAPL

Training: 2020-01-01 to 2020-12-31, Testing: 2020-12-31 to 2021-03-01
Skipping due to insufficient data

Training: 2020-03-01 to 2021-03-01, Testing: 2021-03-01 to 2021-04-30
Skipping due to insufficient data

Training: 2020-04-30 to 2021-04-30, Testing: 2021-04-30 to 2021-06-29
Skipping due to insufficient data

Training: 2020-06-29 to 2021-06-29, Testing: 2021-06-29 to 2021-08-28
Skipping due to insufficient data

Training: 2020-08-28 to 2021-08-28, Testing: 2021-08-28 to 2021-10-27
Skipping due to insufficient data

Training: 2020-10-27 to 2021-10-27, Testing: 2021-10-27 to 2021-12-26
Skipping due to insufficient data

Training: 2020-12-26 to 2021-12-26, Testing: 2021-12-26 to 2022-02-24
Skipping due to insufficient data

Training: 2021-02-24 to 2022-02-24, Testing: 2022-02-24 to 2022-04-25
Skipping due to insufficient data

Training: 2021-04-25 to 2022-04-25, Testing: 2022-04-25 to 2022-06-24
Skipping due to insufficient data

Training: 2021-06-24 to 2022-06-24, 

In [16]:
import pandas as pd

metrics = pd.read_csv("/content/drive/MyDrive/QuantConnect/results_xgb_walkforward_results/xgb_walkforward_metrics.csv")
metrics[metrics['Ticker'] == 'AAPL']


,Ticker,Train Period,Test Period,Model,Accuracy,Sharpe,Drawdown,Return,Final_Portfolio
0,AAPL,2021-10-22 to 2022-10-22,2022-10-22 to 2022-12-21,XGBoost,0.3887,-2.659,2.08,-1.65,98354.48
1,AAPL,2021-12-21 to 2022-12-21,2022-12-21 to 2023-02-19,XGBoost,0.3857,-0.782,0.62,-0.32,99677.43
2,AAPL,2022-02-19 to 2023-02-19,2023-02-19 to 2023-04-20,XGBoost,0.4216,0.513,0.44,0.20,100202.16
3,AAPL,2022-10-17 to 2023-10-17,2023-10-17 to 2023-12-16,XGBoost,0.8215,0.991,0.04,0.06,100059.34
